In [10]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

# ── Paste these in — required to rebuild the model ────────────────────────────
domain2id    = {'nfl': 0, 'financial': 1, 'politics': 2, 'videogames': 3, 'none': 4}
id2domain    = {v: k for k, v in domain2id.items()}
sentiment2id = {'negative': 0, 'positive': 1, "neutral": 2}
id2sentiment = {v: k for k, v in sentiment2id.items()}

# ── Recreate the architecture ─────────────────────────────────────────────────
class MultiTaskModel(nn.Module):
    def __init__(self, backbone, num_domains, num_sentiments, dropout=0.3):
        super().__init__()
        self.backbone = backbone
        hidden = backbone.config.hidden_size

        self.domain_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, num_domains)
        )
        self.sentiment_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, num_sentiments)
        )

    def forward(self, input_ids, attention_mask, labels=None, sentiment_labels=None):
        outputs   = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = outputs.last_hidden_state[:, 0, :]

        domain_logits    = self.domain_head(cls_token)
        sentiment_logits = self.sentiment_head(cls_token)

        loss = None
        if labels is not None and sentiment_labels is not None:
            loss = (
                torch.nn.functional.cross_entropy(domain_logits, labels) +
                torch.nn.functional.cross_entropy(sentiment_logits, sentiment_labels)
            )

        return {
            'loss':             loss,
            'domain_logits':    domain_logits,
            'sentiment_logits': sentiment_logits,
        }

# ── Device ────────────────────────────────────────────────────────────────────
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# ── Load weights ──────────────────────────────────────────────────────────────
backbone  = AutoModel.from_pretrained('distilbert-base-uncased')
model = MultiTaskModel(backbone, num_domains=5, num_sentiments=3)
model.load_state_dict(torch.load('./multitask-classifier/model_weights.pt', map_location=device, weights_only=True))
model.to(device).eval()

tokenizer = AutoTokenizer.from_pretrained('./multitask-classifier')
print("Model loaded and ready")

Using device: mps
Model loaded and ready


In [12]:
import pandas as pd
import torch.nn.functional as F
from sklearn.metrics import classification_report

videogames_df = pd.read_csv("/Users/alecxszhang/Desktop/Stat 359/data/videogames_train.csv")
videogames_df['sentiment'] = videogames_df['sentiment'].str.lower()
videogames_df['domain'] = 'videogames'
# peek at the structure first
print(videogames_df.head())
print(videogames_df.columns.tolist())

      id         topic sentiment  \
0   7871     MaddenNFL  negative   
1   1257   Battlefield  positive   
2  12597  WorldOfCraft   neutral   
3   3107         Dota2   neutral   
4    544   ApexLegends   neutral   

                                                text      domain  
0   The speed this year is horrendous. @ EAMaddenNFL  videogames  
1  . Phantom Bow + Suppressed Deagle - Battlefiel...  videogames  
2  Check out this item I just got! along With the...  videogames  
3  ARTEEZY Fast Game No Mercy played with his Sup...  videogames  
4                  I'm consistently getting top 5 in  videogames  
['id', 'topic', 'sentiment', 'text', 'domain']


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import classification_report

# ── Batch predict ──────────────────────────────────────────────────────────────
def batch_predict(texts, batch_size=64):
    all_results = []
    for i in range(0, len(texts), batch_size):
        batch  = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        for j in range(len(batch)):
            sentiment_probs = F.softmax(outputs['sentiment_logits'][j], dim=-1)
            all_results.append({
                'text':            batch[j],
                'sentiment':       id2sentiment[sentiment_probs.argmax().item()],
                'sentiment_score': round(sentiment_probs.max().item(), 3),
            })

        if (i // batch_size) % 20 == 0:
            print(f"  Processed {min(i + batch_size, len(texts)):,} / {len(texts):,}", end='\r')

    print(f"  Done — processed {len(texts):,} texts")
    return all_results

# ── Evaluate ───────────────────────────────────────────────────────────────────
texts                 = videogames_df['text'].astype(str).tolist()
true_sentiment_labels = videogames_df['sentiment'].map(sentiment2id).tolist()

results               = batch_predict(texts)
pred_sentiment_labels = [sentiment2id[r['sentiment']] for r in results]

print("\n=== Sentiment Classification Report (videogames) ===")
print(classification_report(
    true_sentiment_labels,
    pred_sentiment_labels,
    target_names=list(sentiment2id.keys())
))

  Done — processed 38,090 texts

=== Sentiment Classification Report ===
              precision    recall  f1-score   support

    negative       0.73      0.82      0.77     13881
    positive       0.75      0.74      0.75     14289
     neutral       0.65      0.53      0.58      9920

    accuracy                           0.72     38090
   macro avg       0.71      0.70      0.70     38090
weighted avg       0.71      0.72      0.71     38090



In [ ]:

videogames_df = pd.read_csv("/Users/alecxszhang/Desktop/Stat 359/data/videogames_train.csv")
videogames_df['sentiment'] = videogames_df['sentiment'].str.lower()
videogames_df['domain'] = 'videogames'
# peek at the structure first
print(videogames_df.head())
print(videogames_df.columns.tolist())

In [15]:
def classify_sentiment(score):
    if score < -0.1:
        return 'negative'
    elif score > 0.1:
        return 'positive'
    else:
        return 'neutral'
financial_df = pd.read_csv('/Users/alecxszhang/Desktop/Stat 359/data/Financial_tweets.csv')

financial_df['sentiment'] = financial_df['score'].apply(classify_sentiment)
financial_df['domain'] = 'financial'
financial_df = financial_df.rename(columns={'full_text': 'text'})

In [16]:
texts                 = financial_df['text'].astype(str).tolist()
true_sentiment_labels = financial_df['sentiment'].map(sentiment2id).tolist()

results               = batch_predict(texts)
pred_sentiment_labels = [sentiment2id[r['sentiment']] for r in results]

print("\n=== Sentiment Classification Report (Financial) ===")
print(classification_report(
    true_sentiment_labels,
    pred_sentiment_labels,
    target_names=list(sentiment2id.keys())
))

  Done — processed 12,420 texts

=== Sentiment Classification Report (Financial) ===
              precision    recall  f1-score   support

    negative       0.79      0.78      0.78      3921
    positive       0.73      0.70      0.71      4041
     neutral       0.67      0.70      0.68      4458

    accuracy                           0.72     12420
   macro avg       0.73      0.72      0.73     12420
weighted avg       0.72      0.72      0.72     12420



In [17]:
politics_df = pd.read_csv("/Users/alecxszhang/Desktop/Stat 359/data/politics_sentiment.csv")
politics_df['domain'] = 'politics'
politics_df = politics_df.rename(columns={'tweet': 'text'})
politics_df['sentiment'] = politics_df['sentiment'].map({0: 'negative', 1: 'positive', 2: 'neutral'})
politics_df.head()

,Unnamed: 0,text,sentiment,domain
0,192,@goodlaura What about Reese dying on #TTSC? An...,negative,politics
1,445,Nasty budget due and my iphone is being sent t...,negative,politics
2,1085,@transbay &quot;SFMTA Budget Proposal Hearing:...,negative,politics
3,1988,"Obama is visiting istanbul today, therefore al...",negative,politics
4,2019,just got back from the funeral of a government...,negative,politics


In [20]:
politics_df_clean = politics_df.dropna(subset=['text', 'sentiment'])
politics_df_clean = politics_df_clean[politics_df_clean['sentiment'].isin(sentiment2id.keys())]

texts                 = politics_df_clean['text'].astype(str).tolist()
true_sentiment_labels = politics_df_clean['sentiment'].map(sentiment2id).tolist()

results               = batch_predict(texts)
pred_sentiment_labels = [sentiment2id[r['sentiment']] for r in results]

print("\n=== Sentiment Classification Report (Politics) ===")
print(classification_report(
    true_sentiment_labels,
    pred_sentiment_labels,
    target_names=list(sentiment2id.keys())
))

  Done — processed 5,451 texts

=== Sentiment Classification Report (Politics) ===
              precision    recall  f1-score   support

    negative       0.89      0.88      0.88      2475
    positive       0.88      0.88      0.88      2420
     neutral       0.87      0.89      0.88       556

    accuracy                           0.88      5451
   macro avg       0.88      0.88      0.88      5451
weighted avg       0.88      0.88      0.88      5451



In [21]:
nfl_df = pd.read_csv("/Users/alecxszhang/Desktop/Stat 359/data/nfl_sentiments.csv")
nfl_df['domain'] = 'nfl'

#filter to only tweets that mention NFL-related keywords
# to remove off-topic scraped content
import re
nfl_keywords = r'\b(nfl|football|touchdown|quarterback|qb|mahomes|brady|receiver|linebacker|' \
               r'packers|chiefs|eagles|cowboys|49ers|ravens|browns|patriots|bills|' \
               r'draft|super bowl|playoffs|offense|defense|interception|fumble|yard|'\
               r'rushing|passing|receiver|tight end|kicker)\b'

nfl_df = nfl_df[nfl_df['text'].str.contains(nfl_keywords, case=False, regex=True)]
print(f"NFL tweets after filtering: {len(nfl_df)}")

NFL tweets after filtering: 2823


/var/folders/2p/rhd39m592jv2kqghpzh8s5r40000gp/T/ipykernel_4035/3831012895.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  nfl_df = nfl_df[nfl_df['text'].str.contains(nfl_keywords, case=False, regex=True)]


In [22]:
nfl_df_clean = nfl_df.dropna(subset=['text', 'sentiment'])
nfl_df_clean = nfl_df_clean[nfl_df_clean['sentiment'].isin(sentiment2id.keys())]

texts                 = nfl_df_clean['text'].astype(str).tolist()
true_sentiment_labels = nfl_df_clean['sentiment'].map(sentiment2id).tolist()

results               = batch_predict(texts)
pred_sentiment_labels = [sentiment2id[r['sentiment']] for r in results]

print("\n=== Sentiment Classification Report (NFL) ===")
print(classification_report(
    true_sentiment_labels,
    pred_sentiment_labels,
    target_names=list(sentiment2id.keys())
))

  Done — processed 2,816 texts

=== Sentiment Classification Report (NFL) ===
              precision    recall  f1-score   support

    negative       0.95      0.94      0.94       943
    positive       0.91      0.94      0.93       592
     neutral       0.95      0.95      0.95      1281

    accuracy                           0.94      2816
   macro avg       0.94      0.94      0.94      2816
weighted avg       0.94      0.94      0.94      2816

